# NB_hospital_daily_summary

Builds `adb_medilake.gold.hospital_daily_summary` per the STTM *Silver to Gold (CareSync Health Network)* v1.0, sheet `hospital_daily_summary`.

* **Grain:** one row per hospital per calendar day (days with at least one lab test)
* **Driving table:** `adb_medilake.silver.lab_results` grouped by hospital_id, test_date
* **Joins:** LEFT JOIN `silver.hospitals` on hospital_id; LEFT JOIN `silver.calendar` on date_key = yyyyMMdd(test_date)
* **Window:** LAG(total_tests) partitioned by hospital_id ordered by summary_date (previous day with activity)
* **Load:** FULL rebuild with CREATE OR REPLACE TABLE ... AS SELECT

Implementation note: `silver.lab_results` contains exact duplicate `lab_result_id` rows, so lab rows are de-duplicated on `lab_result_id` (latest `update_timestamp`) before aggregating.

In [0]:
%sql
-- Full rebuild of gold.hospital_daily_summary (one row per hospital per day)
CREATE OR REPLACE TABLE adb_medilake.gold.hospital_daily_summary
COMMENT 'Hospital daily lab activity summary with day-over-day trend (STTM silver to gold v1.0)'
AS
WITH lab AS (
  -- De-duplicate lab results to one row per lab_result_id
  SELECT *
  FROM adb_medilake.silver.lab_results
  QUALIFY row_number() OVER (PARTITION BY lab_result_id ORDER BY update_timestamp DESC) = 1
),
daily AS (
  SELECT
    hospital_id,
    test_date                             AS summary_date,
    COUNT(*)                              AS total_tests,
    COUNT(DISTINCT patient_id)            AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)             AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')  AS critical_test_count
  FROM lab
  GROUP BY hospital_id, test_date
),
trend AS (
  SELECT
    *,
    LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests
  FROM daily
)
SELECT
  t.hospital_id,
  h.hospital_name,
  t.summary_date,
  c.day_name,
  c.is_weekend,
  CAST(t.total_tests AS INT)                                        AS total_tests,
  CAST(t.unique_patients_tested AS INT)                             AS unique_patients_tested,
  CAST(t.active_doctor_count AS INT)                                AS active_doctor_count,
  CAST(t.abnormal_test_count AS INT)                                AS abnormal_test_count,
  CAST(t.critical_test_count AS INT)                                AS critical_test_count,
  CAST(t.abnormal_test_count * 100.0 / t.total_tests AS DECIMAL(5,2)) AS abnormal_rate_pct,
  CAST(t.critical_test_count * 100.0 / t.total_tests AS DECIMAL(5,2)) AS critical_rate_pct,
  CAST(t.prev_day_total_tests AS INT)                               AS prev_day_total_tests,
  CAST(
    CASE WHEN t.prev_day_total_tests IS NULL OR t.prev_day_total_tests = 0 THEN NULL
         ELSE (t.total_tests - t.prev_day_total_tests) * 100.0 / t.prev_day_total_tests
    END AS DECIMAL(6,2))                                            AS test_volume_change_pct
FROM trend t
LEFT JOIN adb_medilake.silver.hospitals h
  ON t.hospital_id = h.hospital_id
LEFT JOIN adb_medilake.silver.calendar c
  ON c.date_key = CAST(date_format(t.summary_date, 'yyyyMMdd') AS INT);

In [0]:
# Return the record count of the rebuilt gold table as the notebook exit value
updated_df = spark.table("adb_medilake.gold.hospital_daily_summary")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))